In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
df = pd.read_csv(r"C:\Users\Prasad\Desktop\secure-bank-fraud-ml\data\paysim_model_ready.csv")

In [5]:
df

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,hour_of_day,day,log1p_amount,amount_to_balance_ratio,is_high_amount,balance_diff_org,balance_error,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
0,150540.16,9912.00,0.00,29817.59,180357.75,0,16,14,11.921992,15.186135,0,9912.00,-1.406282e+05,True,False,False,False,0
1,66723.64,0.00,0.00,1136277.81,1203001.45,0,17,11,11.108330,66723.640000,0,0.00,-6.672364e+04,True,False,False,False,0
2,1039375.01,2328.00,0.00,437583.33,1476958.34,0,11,9,13.854131,446.275230,0,2328.00,-1.037047e+06,False,False,False,True,0
3,9178.61,96237.62,87059.01,0.00,0.00,0,11,1,9.124740,0.095373,0,9178.61,0.000000e+00,False,False,True,False,0
4,4527.24,51925.00,47397.76,0.00,0.00,0,23,1,8.418089,0.087186,0,4527.24,-1.818989e-12,False,False,True,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636257,184745.77,1155.00,0.00,186549.97,371295.74,0,18,28,12.126741,159.814680,0,1155.00,-1.835908e+05,True,False,False,False,0
636258,78836.94,30658.00,0.00,4642989.02,4721825.96,0,20,0,11.275150,2.571413,0,30658.00,-4.817894e+04,True,False,False,False,0
636259,265300.00,11400.00,276700.00,0.00,488175.01,0,15,0,12.488620,23.269889,0,-265300.00,-5.306000e+05,False,False,False,False,0
636260,6747.46,6467.72,0.00,0.00,0.00,0,11,8,8.817070,1.043090,0,6467.72,-2.797400e+02,False,False,True,False,0


In [6]:
X = df.drop("isFraud", axis=1)
y =df['isFraud']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1
)

lr.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score, roc_curve


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    print(classification_report(y_test, y_pred))
    print("ROC-AUC", roc_auc_score(y_test, y_prob))

In [13]:
evaluate_model(lr, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      0.97      0.98    127089
           1       0.03      0.90      0.06       164

    accuracy                           0.97    127253
   macro avg       0.52      0.93      0.52    127253
weighted avg       1.00      0.97      0.98    127253

ROC-AUC 0.9769233160782851


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
evaluate_model(rf, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    127089
           1       1.00      0.99      1.00       164

    accuracy                           1.00    127253
   macro avg       1.00      1.00      1.00    127253
weighted avg       1.00      1.00      1.00    127253

ROC-AUC 1.0


In [15]:
pip install lightgbm


Note: you may need to restart the kernel to use updated packages.


In [16]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimator=200,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=42
)

lgbm.fit(X_train, y_train)
evaluate_model(lgbm, X_test, y_test)

[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Info] Number of positive: 657, number of negative: 508352
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004762 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2360
[LightGBM] [Info] Number of data points in the train set: 509009, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
         

In [17]:
leakage_cols = [
    'balance_diff_org',
    'balance_error',
    'amount_to_balance_ratio'
]
x_no_leak = X.drop(columns=leakage_cols)

In [28]:
df_sorted = df.sort_values('day')

train = df_sorted.iloc[:int(0.8 * len(df_sorted))]
test = df_sorted.iloc[int(0.8 * len(df_sorted)):]

X_train = train.drop('isFraud', axis=1)
y_train = train['isFraud']

X_test = test.drop('isFraud', axis=1)
y_test = test['isFraud']

In [25]:
lr.fit(X_train, y_train)
evaluate_model(lr, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      0.98      0.99    126836
           1       0.11      0.88      0.20       417

    accuracy                           0.98    127253
   macro avg       0.55      0.93      0.59    127253
weighted avg       1.00      0.98      0.99    127253

ROC-AUC 0.9893407737463882


In [26]:
rf.fit(X_train, y_train)
evaluate_model(rf, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    126836
           1       1.00      1.00      1.00       417

    accuracy                           1.00    127253
   macro avg       1.00      1.00      1.00    127253
weighted avg       1.00      1.00      1.00    127253

ROC-AUC 1.0


In [27]:
lgbm.fit(X_train, y_train)
evaluate_model(lgbm, X_test, y_test)

[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Info] Number of positive: 404, number of negative: 508605
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005580 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2344
[LightGBM] [Info] Number of data points in the train set: 509009, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

In [29]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    random_state=42
)



In [30]:
lgbm.fit(X_train, y_train)
evaluate_model(lgbm, X_test, y_test)

[LightGBM] [Info] Number of positive: 404, number of negative: 508605
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005728 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2344
[LightGBM] [Info] Number of data points in the train set: 509009, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai